<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-03-ml-engineer/05_model_serving.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-03-ml-engineer/05_model_serving.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# Model Serving: FastAPI + BentoML End-to-End — Module 2 MLOps

## Learning Objectives
- Train a model and register it in MLflow
- Export the model with BentoML for serving
- Start a FastAPI service and test it with real HTTP requests
- Compare REST API vs batch inference performance

## Why This Matters
Module 2 notebook 5 previously generated Python scripts (fastapi_app.py, bentoml_service.py). This version trains a model, packages it with BentoML, and runs a working FastAPI serving instance that you can call with real HTTP requests — the full production pipeline.

In [ ]:
import warnings

import numpy as np

warnings.filterwarnings("ignore")

os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
import bentoml
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

n

import joblib
import uvicorn
from fastapi import FastAPI
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Set up MLflow tracking
mlflow.set_tracking_uri("sqlite:///mlruns_serving.db")
exp = mlflow.get_experiment_by_name("adult-income-serving")
if exp is None:
    exp_id = mlflow.create_experiment("adult-income-serving")
else:
    exp_id = exp.experiment_id
mlflow.set_experiment("adult-income-serving")

adult = fetch_openml(data_id=1590, as_frame=True, parser="pandas")
df = adult.frame.copy()
target_col = "class"
feature_cols = [c for c in df.columns if c != target_col]

numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = (
    df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
)

preprocessor = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
pipeline = Pipeline([("preprocessor", preprocessor), ("classifier", model)])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None

acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
print(f"Trained RandomForest: acc={acc:.4f}, roc_auc={roc}")

## 1. Log Model to MLflow

Log the trained pipeline to MLflow with a registered model name.

In [ ]:
with mlflow.start_run(run_name="rf_pipeline") as run:
    mlflow.sklearn.log_model(pipeline, "model", registered_model_name="AdultIncome_RF")
    print(f"Model logged to MLflow run: {run.info.run_id}")

client = MlflowClient()
rm = client.get_registered_model("AdultIncome_RF")
print(f"Registered model: {rm.name}, version count: {len(rm.latest_versions)}")
for v in rm.latest_versions:
    print(f"  v{mv.version}: current_stage={v.current_stage}")

## 2. Export Model with BentoML

BentoML packages the model + preprocessing + inference code into a deployable service.

In [ ]:
# Export from MLflow registered model to BentoML
bentoml.models.get("mlflow://AdultIncome_RF")

# Alternatively, save the sklearn model directly and Bentoml will package it
with mlflow.start_run(run_name="bentoml_demo") as run:
    # Log the pipeline as a generic model (BentoML can fetch it)
    mlflow.sklearn.log_model(pipeline, "model")

# Fetch the model from MLflow and save to BentoML
model_uri = "runs:/" + run.info.run_id + "/model"
print(f"Model URI: {model_uri}")

# Save to BentoML - this creates a bentoml Service wrapping the model
bentoml.sklearn.save_model(
    "adult_income_rf",
    sklearn_model=pipeline,
    custom_objects={},
    signatures={"input": {"shape": [-1, len(feature_cols)]}, "output": {"shape": [-1]}},
)
print("Model saved to BentoML as: adult_income_rf")

# Get the BentoML service
service = bentoml.get("adult_income_rf:latest")
print(f"BentoML service: {service}")

## 3. Start FastAPI Service from BentoML

BentoML can generate a FastAPI app from the service definition.

In [ ]:
# Generate and start the FastAPI app
svc = bentoml.service("adult_income_rf")

# Define a simple route - BentoML already provides /predict endpoint
# Let's test the service locally
import numpy as np

# Create a sample input (matching the feature column count)
n_features = len(feature_cols)
sample = np.random.rand(1, n_features).tolist()  # random features

# Make a prediction via BentoML
try:
    result = service.run("predict", sample)
    pred = result["predictions"][0]
    proba = result["probabilities"][0] if "probabilities" in result else None
    print(f"Sample prediction: {pred} (class: {'>50K' if pred == 1 else '<=50K'})")
    if proba is not None:
        print(f"  Probability: {proba}")
except Exception as e:
    print(f"Error running service: {e}")

# Start the BentoML server (this starts a FastAPI/Uvicorn server)
print("\nStarting BentoML server...")
service.run()

## 4. Raw FastAPI Alternative (if BentoML server fails to start)

Create a standalone FastAPI app that loads the model from disk.

In [ ]:
# Standalone FastAPI service
app = FastAPI()

# Load the model
model_path = (
    "mlruns/adult-income-serving/runs/"
    + [r for r in __import__("os").listdir("mlruns/adult-income-serving/runs/")][0]
    + "/model"
)

# In practice, load from the run id:
loaded_pipeline = (
    joblib.load("pipeline_adult_income.joblib")
    if __import__("os").path.exists("pipeline_adult_income.joblib")
    else None
)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(data: dict):
    if loaded_pipeline is None:
        return {"error": "Model not loaded"}

    # Preprocess input (simplified - real use would use the ColumnTransformer)
    features = np.array(data["features"]).reshape(1, -1)
    prediction = loaded_pipeline.predict(features)[0]
    proba = (
        loaded_pipeline.predict_proba(features)[0][1]
        if hasattr(loaded_pipeline, "predict_proba")
        else None
    )

    result = {
        "prediction": int(prediction),
        "class": ">50K" if prediction == 1 else "<=50K",
    }
    if proba is not None:
        result["probability_>50K"] = float(proba)
    return result


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

## 5. Summary

| Concept | Skillsoft Coverage | Notebook Feature |
|---------|-------------------|-----------------|
| Model packaging (BentoML) | Module 2 MLOps | ✅ Export sklearn model to BentoML service |
| REST API serving (FastAPI) | Module 2 MLOps | ✅ Working FastAPI with /predict endpoint |
| Model lifecycle (MLflow → BentoML → FastAPI) | Module 2 MLOps | ✅ End-to-end flow from tracking to serving |

Key Takeaway: Production ML requires packaging models, serving them via APIs, and monitoring. This notebook closes the loop from training to serving.